# 05 -- Final Load Preparation for Tableau

**Project:** SuperStore Retail Performance & Profitability Analysis  
**Workflow:** Final QA and export for Tableau Public  
**Goal:** Confirm that the final CSV is Tableau-ready, professionally documented, and limited to the approved 37 columns.

## 1. Environment Setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "SuperStore_Analysis/Tableau_Analysis/scripts/superstore_pipeline.py").exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the repository root from the notebook environment.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from SuperStore_Analysis.Tableau_Analysis.scripts.superstore_pipeline import (
    FINAL_SCHEMA,
    LOYAL_CUSTOMER_MIN_PURCHASES,
    build_clean_dataset,
    build_tableau_ready_dataset,
    compute_project_metrics,
    copy_raw_snapshot,
    export_pipeline_outputs,
    load_existing_cleaned_dataset,
    load_raw_dataset,
    prepare_analysis_frame,
    resolve_project_paths,
    validate_final_schema,
)

PATHS = resolve_project_paths(REPO_ROOT / "SuperStore_Analysis" / "Tableau_Analysis")
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


## 2. Load the Final Cleaned Dataset

In [2]:
df_clean = pd.read_csv(PATHS.cleaned_output_path)
df_tableau = build_tableau_ready_dataset(df_clean)
display(df_tableau.head(8))


,Row ID,Transaction Id (PK),Order ID,Order Date,Year,Month,Quarter,Ship Date,Shipping Delay,Shipping Speed,Ship Mode,Customer ID,Customer Name,Customer Type,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Order-Size,Sales Per Unit,Discount,Profit,Profit Margin,Loss Severity,Loss Flag,Order-Level Total Sales (Grouped by Order ID C),Customer Purchase Frequency (Customer ID = L),Total sales per customer,Discount Amount (Sales × Discount)
0,1,TXN-000001,CA-2016-152156,11/8/2016,2016,November,Q4,11/11/2016,3,Fast,Second Class,CG-12520,Claire Gute,Occasional,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,FURNITURE,Bookcases,Bush Somerset Collection Bookcase,261.96,2,Medium,130.98,0%,41.91,0.16,Profit,Profit,993.90,5,"1,148.78",0.00
1,2,TXN-000002,CA-2016-152156,11/8/2016,2016,November,Q4,11/11/2016,3,Fast,Second Class,CG-12520,Claire Gute,Occasional,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,FURNITURE,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,Large,243.98,0%,219.58,0.30,Profit,Profit,993.90,5,"1,148.78",0.00
2,3,TXN-000003,CA-2016-138688,6/12/2016,2016,June,Q2,6/16/2016,4,Normal,Second Class,DV-13045,Darrin Van Huff,Occasional,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,OFFICE SUPPLIES,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,Small,7.31,0%,6.87,0.47,Profit,Profit,14.62,9,"1,119.48",0.00
3,4,TXN-000004,US-2015-108966,10/11/2015,2015,October,Q4,10/18/2015,7,Slow,Standard Class,SO-20335,Sean O'Donnell,Loyal,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,FURNITURE,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,Large,191.52,45%,-383.03,-0.40,High Loss,Loss,979.95,15,"2,602.58",430.91
4,5,TXN-000005,US-2015-108966,10/11/2015,2015,October,Q4,10/18/2015,7,Slow,Standard Class,SO-20335,Sean O'Donnell,Loyal,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,OFFICE SUPPLIES,Storage,Eldon Fold 'N Roll Cart System,22.37,2,Small,11.18,20%,2.52,0.11,Profit,Profit,979.95,15,"2,602.58",4.47
5,6,TXN-000006,CA-2014-115812,6/9/2014,2014,June,Q2,6/14/2014,5,Normal,Standard Class,BH-11710,Brosina Hoffman,Loyal,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,FURNITURE,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.86,7,Small,6.98,0%,14.17,0.29,Profit,Profit,"3,714.29",24,"6,255.33",0.00
6,7,TXN-000007,CA-2014-115812,6/9/2014,2014,June,Q2,6/14/2014,5,Normal,Standard Class,BH-11710,Brosina Hoffman,Loyal,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,OFFICE SUPPLIES,Art,Newell 322,7.28,4,Small,1.82,0%,1.97,0.27,Profit,Profit,"3,714.29",24,"6,255.33",0.00
7,8,TXN-000008,CA-2014-115812,6/9/2014,2014,June,Q2,6/14/2014,5,Normal,Standard Class,BH-11710,Brosina Hoffman,Loyal,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,TECHNOLOGY,Phones,Mitel 5320 IP Phone VoIP phone,907.15,6,Large,151.19,20%,90.72,0.10,Profit,Profit,"3,714.29",24,"6,255.33",181.43


## 3. Final Schema Validation

In [3]:
schema_check = validate_final_schema(df_tableau)
metrics = compute_project_metrics(df_tableau)
validation_table = pd.DataFrame({
    'Check': ['Matches approved schema', 'Row count', 'Column count', 'Duplicate Transaction IDs', 'Missing values'],
    'Value': [
        schema_check['matches_expected_schema'],
        len(df_tableau),
        df_tableau.shape[1],
        int(df_tableau['Transaction Id (PK)'].duplicated().sum()),
        int(df_tableau.isna().sum().sum()),
    ],
})
display(validation_table)
print(schema_check)
print(metrics)


,Check,Value
0,Matches approved schema,True
1,Row count,9993
2,Column count,37
3,Duplicate Transaction IDs,0
4,Missing values,0


{'matches_expected_schema': True, 'missing_columns': [], 'unexpected_columns': []}
{'rows': 9993, 'sales_total': 2296919.28, 'profit_total': 286408.6, 'profit_margin': 0.1247, 'loss_transactions': 1870, 'loss_transaction_pct': 0.1871, 'high_discount_transactions': 1392, 'high_discount_loss_transactions': 1347, 'unique_orders': 5009, 'unique_customers': 793, 'unique_products': 1862, 'total_quantity': 37871}


## 4. Field Audit

In [4]:
field_audit = pd.DataFrame({
    'Column': df_tableau.columns,
    'Data Type': df_tableau.dtypes.astype(str).values,
    'Missing Values': df_tableau.isna().sum().values,
    'Sample Value': [df_tableau[column].dropna().iloc[0] if not df_tableau[column].dropna().empty else None for column in df_tableau.columns],
})
display(field_audit)


,Column,Data Type,Missing Values,Sample Value
0,Row ID,int64,0,1
1,Transaction Id (PK),object,0,TXN-000001
2,Order ID,object,0,CA-2016-152156
3,Order Date,object,0,11/8/2016
4,Year,int64,0,2016
5,Month,object,0,November
6,Quarter,object,0,Q4
7,Ship Date,object,0,11/11/2016
8,Shipping Delay,int64,0,3
9,Shipping Speed,object,0,Fast


## 5. Tableau Calculated Fields to Create in the Workbook

The dataset intentionally excludes helper columns such as sort keys, upper-case copies, and profit splits. Create them inside Tableau only if you need them for a visual.

In [5]:
tableau_fields = pd.DataFrame({
    'Calculated Field': [
        'Discount Rate',
        'Profit Margin %',
        'Transaction Count',
        'Loss Transactions',
        'High Discount Transactions',
        'Profit Only',
        'Loss Only',
        'Year Quarter Label',
        'Month Sort',
    ],
    'Suggested Tableau Logic': [
        'FLOAT(REPLACE([Discount], "%", "")) / 100',
        'SUM([Profit]) / SUM([Sales])',
        'COUNT([Transaction Id (PK)])',
        'SUM(IIF([Loss Flag] = "Loss", 1, 0))',
        'SUM(IIF(FLOAT(REPLACE([Discount], "%", "")) / 100 > 0.20, 1, 0))',
        'IF [Profit] > 0 THEN [Profit] ELSE 0 END',
        'IF [Profit] < 0 THEN [Profit] ELSE 0 END',
        'STR([Year]) + " " + [Quarter]',
        'DATEPART("month", DATE([Order Date]))',
    ],
})
display(tableau_fields)


,Calculated Field,Suggested Tableau Logic
0,Discount Rate,"FLOAT(REPLACE([Discount], ""%"", """")) / 100"
1,Profit Margin %,SUM([Profit]) / SUM([Sales])
2,Transaction Count,COUNT([Transaction Id (PK)])
3,Loss Transactions,"SUM(IIF([Loss Flag] = ""Loss"", 1, 0))"
4,High Discount Transactions,"SUM(IIF(FLOAT(REPLACE([Discount], ""%"", """")) / ..."
5,Profit Only,IF [Profit] > 0 THEN [Profit] ELSE 0 END
6,Loss Only,IF [Profit] < 0 THEN [Profit] ELSE 0 END
7,Year Quarter Label,"STR([Year]) + "" "" + [Quarter]"
8,Month Sort,"DATEPART(""month"", DATE([Order Date]))"


## 6. Export the Final Tableau File

In [6]:
_, df_tableau_exported, dropped_row_ids = export_pipeline_outputs(paths=PATHS)
export_status = pd.DataFrame({
    'Output': ['Final Tableau-ready CSV'],
    'Path': [str(PATHS.tableau_output_path)],
    'Rows': [len(df_tableau_exported)],
    'Columns': [df_tableau_exported.shape[1]],
    'Dropped Duplicate Row IDs': [', '.join(map(str, dropped_row_ids)) if dropped_row_ids else 'None'],
})
display(export_status)


,Output,Path,Rows,Columns,Dropped Duplicate Row IDs
0,Final Tableau-ready CSV,/Users/ravleensingh/Documents/Capstone/DVA_Cap...,9993,37,3407


## 7. Final Load Notes

- `superstore_tableau_ready_dataset.csv` is now limited to the exact approved 37 columns.
- Analysis helpers are kept in notebooks and should be recreated in Tableau only when needed.
- The dataset is ready for the final multi-page Tableau Public dashboard build.